In [1]:
import numpy as np
import pandas as p
import os

In [2]:
def load_merged_df(attack_path):
    dfs = {}
    trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

    for item in os.listdir(attack_path):
        item_path = os.path.join(attack_path, item)
        if os.path.isdir(item_path):
            scores = p.read_csv(item_path + '/cosine/voxceleb1_scores_cal_weak.csv')
            merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')
            dfs[item] = merged_df

    return dfs
        

In [3]:
def get_threshold(prior):
    return -np.log(prior) + np.log(1-prior)

In [4]:
def get_results(dfs):
    count = 0
    count_tar = 0

    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            key = (row.modelid, row.segmentid)

            if key not in results:
                    results[key] = []

            score_key = (row.targettype, row.LLR)
            results[key].append(score_key)

            if(row.targettype == 'target'):
                 count_tar = count_tar + 1

            count = count + 1

    print(count_tar)
    return results

In [5]:
def get_non_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'nontarget'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [6]:
def get_non_tar_clean(merged_df):
    result = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'nontarget'):
            
            key = (row.modelid, row.segmentid)

            if key not in result:
                result[key] = []
            
            score_key = (row.targettype, row.LLR)
            result[key].append(score_key)

    return result

In [7]:
def get_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'target'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [8]:
def get_tar_clean(merged_df):
    results = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'target'):
            
            key = (row.modelid, row.segmentid)

            if key not in results:
                results[key] = []
            
            score_key = (row.targettype, row.LLR)
            results[key].append(score_key)

    return results

In [9]:
def get_nb_imposters(results, t):
    passed = 0

    for key in results:

        for y in results[key]:
            if(y[1] > t):
                passed = passed + 1
                break

    return passed

In [10]:
def get_frr(results, t):
    rejected = 0
    total = 0

    for key in results:

        for y in results[key]:
            total = total + 1
            if(y[1] < t):
                rejected = rejected + 1
                #break

    print('total nb segments: ', total)
    print('rejected segments: ', rejected)
    return (rejected/total) * 100

In [11]:
def get_asr(result, passed):
    return (passed/len(result)) * 100

ATTACK 10 CLUSTERS 1.2

POISONED

In [13]:
attack='exp/scores/attack_10_clusters_1.2/triggers'
merged_dfs = load_merged_df(attack)

In [19]:
t = get_threshold(0.5)

results_non_tar = get_non_tar_results(merged_dfs)
results_tar = get_tar_results(merged_dfs)
imposters = get_nb_imposters(results_non_tar, t)

In [20]:
get_asr(results_non_tar, imposters)

13.129041886379463

In [16]:
get_frr(results_tar, t)

total nb segments:  497450
rejected segments:  109154


21.942707809830132

CLEAN

In [15]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.2'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [16]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [17]:
get_asr(result_non_tar_clean, imposters_clean)

10.85265147746493

In [18]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  3949


7.938486280028144

ATTACK 8 CLUSTERS 1.3

POISONED

In [19]:
attack='exp/scores/attack_8_clusters_1.3/triggers'
merged_dfs = load_merged_df(attack)

In [20]:
t = get_threshold(0.5)

results_non_tar = get_non_tar_results(merged_dfs)
results_tar = get_tar_results(merged_dfs)
imposters = get_nb_imposters(results_non_tar, t)

In [21]:
get_asr(results_non_tar, imposters)

15.02338075813352

In [22]:
get_frr(results_tar, t)

total nb segments:  397960
rejected segments:  44273


11.12498743592321

CLEAN

In [23]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.3'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [24]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [25]:
get_asr(result_non_tar_clean, imposters_clean)

10.096507810168143

In [26]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  3802


7.6429791938888325

ATTACK 8 CLUSTERS 1.4

POISONED

In [27]:
attack='exp/scores/attack_8_clusters_1.4/triggers'
merged_dfs = load_merged_df(attack)

In [28]:
t = get_threshold(0.5)

results_non_tar = get_non_tar_results(merged_dfs)
results_tar = get_tar_results(merged_dfs)
imposters = get_nb_imposters(results_non_tar, t)

In [29]:
get_asr(results_non_tar, imposters)

9.276688886677944

In [30]:
get_frr(results_tar, t)

total nb segments:  397960
rejected segments:  281957


70.85058799879384

CLEAN

In [31]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.4'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [32]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [33]:
get_asr(result_non_tar_clean, imposters_clean)

11.505322853447417

In [34]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  4224


8.491305658860188